# 02 — Nettoyage, standardisation et fusion du dataset

## Projet : Observatoire Data Science & IA

Ce notebook documente la partie **Data Engineering** du projet :

1. Chargement du dataset final propre.
2. Explication du schéma commun utilisé pour fusionner les sources.
3. Vérification des colonnes essentielles.
4. Contrôle des dates, pays, compétences et sources.
5. Vérification des valeurs manquantes et des doublons.
6. Sauvegarde de la version finale nettoyée.

> Remarque : le notebook de travail initial contenait toutes les étapes détaillées de filtrage source par source. Ce notebook est une version propre et présentable qui résume et vérifie la fusion finale.

## 1. Importation des bibliothèques

In [3]:
import warnings

warnings.filterwarnings("ignore")

In [4]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Chargement du dataset final nettoyé

Le fichier utilisé ici est :

`final_data_ai_jobs_clean_2020_2026.csv`

Il correspond au dataset consolidé après collecte, filtrage Data/AI, standardisation des colonnes et nettoyage global.

In [5]:
DATA_PATH = "final_data_ai_jobs_clean_2020_2026.csv"

df_clean = pd.read_csv(DATA_PATH)

print("Nombre de lignes :", df_clean.shape[0])
print("Nombre de colonnes :", df_clean.shape[1])
df_clean.head(3).T

Nombre de lignes : 751801
Nombre de colonnes : 30


,0,1,2
source,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset
platform,Synthetic / Global,Synthetic / Global,Synthetic / Global
job_id,1,2,3
job_title,AI Researcher,MLOps Engineer,Data Analyst
company_name,NaN,NaN,NaN
country,Canada,India,United Kingdom
city,Berlin,Tokyo,Bangalore
date_posted,2021-04-01 00:00:00,2020-04-12 00:00:00,2023-01-31 00:00:00
year,2021,2020,2023
month,4.0,4.0,1.0


## 3. Schéma commun utilisé pour la fusion

Les fichiers sources avaient des noms de colonnes différents. Par exemple :

- `titre_offre`, `title`, `job_title` → `job_title`
- `pays`, `country`, `job_country` → `country`
- `date_reference`, `posting_date`, `job_posted_date` → `date_posted`
- `competences_detectees`, `skills`, `job_skills`, `skills_tech_stack` → `skills`

Pour permettre la fusion, toutes les sources ont été ramenées vers un schéma commun.

In [6]:
standard_columns = [
    "source",
    "platform",
    "job_id",
    "job_title",
    "company_name",
    "country",
    "city",
    "date_posted",
    "year",
    "month",
    "year_month",
    "job_type",
    "remote_status",
    "experience_level",
    "education_required",
    "industry",
    "category",
    "description",
    "skills",
    "tools_used",
    "salary",
    "salary_currency",
    "job_url",
    "original_file"
]

print("Nombre de colonnes attendues :", len(standard_columns))

missing_cols = [col for col in standard_columns if col not in df_clean.columns]
extra_cols = [col for col in df_clean.columns if col not in standard_columns]

print("Colonnes manquantes par rapport au schéma commun :", missing_cols)
print("Colonnes supplémentaires :", extra_cols)

Nombre de colonnes attendues : 24
Colonnes manquantes par rapport au schéma commun : []
Colonnes supplémentaires : ['country_clean', 'skills_clean', 'skills_original', 'remote_status_clean', 'remote_status_original', 'job_category_clean']


## 4. Vérification des types et des formats

La colonne `date_posted` doit être convertie en format datetime pour permettre les analyses temporelles.

In [7]:
df_clean["date_posted"] = pd.to_datetime(df_clean["date_posted"], errors="coerce")

print("Date minimale :", df_clean["date_posted"].min())
print("Date maximale :", df_clean["date_posted"].max())
print("Dates manquantes :", df_clean["date_posted"].isna().sum())

print("\nTypes des colonnes :")
df_clean.dtypes

Date minimale : 2020-01-01 00:00:00
Date maximale : 2026-05-06 16:47:40
Dates manquantes : 1240

Types des colonnes :


source                            object
platform                          object
job_id                            object
job_title                         object
company_name                      object
country                           object
city                              object
date_posted               datetime64[ns]
year                               int64
month                            float64
year_month                        object
job_type                          object
remote_status                     object
experience_level                  object
education_required                object
industry                          object
category                          object
description                       object
skills                            object
tools_used                        object
salary                            object
salary_currency                   object
job_url                           object
original_file                     object
country_clean   

## 5. Vérification des colonnes essentielles

Les colonnes essentielles pour notre projet sont :

- `job_title`
- `country`
- `date_posted`
- `skills`
- `source`
- `platform`
- `original_file`

Elles sont nécessaires pour analyser les offres par métier, pays, date, compétence et source.

In [8]:
essential_columns = [
    "job_title",
    "country",
    "date_posted",
    "skills",
    "source",
    "platform",
    "original_file"
]

essential_quality = pd.DataFrame({
    "missing_count": df_clean[essential_columns].isna().sum(),
    "missing_percent": (df_clean[essential_columns].isna().mean() * 100).round(3)
})

essential_quality

,missing_count,missing_percent
job_title,0,0.000
country,265,0.035
date_posted,1240,0.165
skills,325,0.043
source,0,0.000
platform,0,0.000
original_file,0,0.000


## 6. Répartition par source après fusion

Cette étape permet de vérifier le poids de chaque fichier source dans le dataset final.

In [9]:
source_distribution = df_clean["original_file"].value_counts().reset_index()
source_distribution.columns = ["original_file", "number_of_jobs"]
source_distribution["percentage"] = (source_distribution["number_of_jobs"] / len(df_clean) * 100).round(2)
source_distribution

,original_file,number_of_jobs,percentage
0,huggingface_global_2023_data_jobs_skills_filtered.csv,622067,82.74
1,global_ai_jobs_dataset.csv,120000,15.96
2,linkedin_us_uk_canada_australia_2024-01-12_to_2024-01-17_data_ai_skills_filtered.csv,5732,0.76
3,linkedin_datastax_2023-12-05_to_2024-04-20_data_ai_filtered.csv,1777,0.24
4,jobs_raw.json,960,0.13
5,kaggle_global_2025_data_science_jobs_skills_filtered.csv,941,0.13
6,jobs_data_ai_clean.csv,274,0.04
7,aijobs_raw.csv,50,0.01


## 7. Répartition temporelle

On vérifie ici la couverture par année et par mois.

In [10]:
jobs_by_year = df_clean["year"].value_counts().sort_index().reset_index()
jobs_by_year.columns = ["year", "number_of_jobs"]
jobs_by_year

,year,number_of_jobs
0,2020,20010
1,2021,20175
2,2022,19872
3,2023,641784
4,2024,27642
5,2025,20985
6,2026,1333


In [11]:
jobs_by_month = df_clean["year_month"].value_counts(dropna=False).sort_index().reset_index()
jobs_by_month.columns = ["year_month", "number_of_jobs"]
jobs_by_month.head(30)

,year_month,number_of_jobs
0,2020-01,1692
1,2020-02,1622
2,2020-03,1633
3,2020-04,1658
4,2020-05,1636
5,2020-06,1603
6,2020-07,1693
7,2020-08,1694
8,2020-09,1652
9,2020-10,1690


## 8. Nettoyage et contrôle des pays

La colonne `country` a été normalisée afin de regrouper les codes et les noms complets. Exemples :

- `US` → `United States`
- `GB` → `United Kingdom`
- `MA` → `Morocco`
- `AE` → `United Arab Emirates`

On vérifie maintenant les pays les plus représentés.

In [12]:
print("Nombre de pays différents :", df_clean["country"].nunique(dropna=True))
print("Pays manquants :", df_clean["country"].isna().sum())

countries = df_clean["country"].value_counts(dropna=False).reset_index()
countries.columns = ["country", "number_of_jobs"]
countries["percentage"] = (countries["number_of_jobs"] / len(df_clean) * 100).round(2)
countries.head(30)

Nombre de pays différents : 164
Pays manquants : 265


,country,number_of_jobs,percentage
0,United States,199743,26.57
1,India,56012,7.45
2,United Kingdom,46123,6.14
3,Germany,33488,4.45
4,France,32273,4.29
5,Singapore,31216,4.15
6,Netherlands,28268,3.76
7,Canada,25601,3.41
8,Australia,21516,2.86
9,Spain,20278,2.70


## 9. Vérification de la colonne `remote_status`

Les valeurs ont été normalisées en catégories simples :

- `Remote`
- `Hybrid`
- `On-site`
- `Not specified`

In [13]:
remote_distribution = df_clean["remote_status"].value_counts(dropna=False).reset_index()
remote_distribution.columns = ["remote_status", "number_of_jobs"]
remote_distribution["percentage"] = (remote_distribution["number_of_jobs"] / len(df_clean) * 100).round(2)
remote_distribution

,remote_status,number_of_jobs,percentage
0,On-site,608492,80.94
1,Remote,100902,13.42
2,Hybrid,40077,5.33
3,Not specified,2330,0.31


## 10. Vérification de la colonne `skills`

La colonne `skills` a été nettoyée pour harmoniser les formats :

- listes Python
- séparateurs `;`, `|`, `/`
- majuscules/minuscules
- espaces inutiles

Exemple : `Python;SQL;Power BI` devient `python, sql, power bi`.

In [14]:
print("Offres avec skills :", df_clean["skills"].notna().sum())
print("Offres sans skills :", df_clean["skills"].isna().sum())
print("Pourcentage avec skills :", round(df_clean["skills"].notna().mean() * 100, 2), "%")

df_clean[["job_title", "country", "year", "skills"]].head(10)

Offres avec skills : 751476
Offres sans skills : 325
Pourcentage avec skills : 99.96 %


,job_title,country,year,skills
0,AI Researcher,Canada,2021,"python, computer vision, sql, nlp"
1,MLOps Engineer,India,2020,"nlp, pytorch, data analysis, computer vision"
2,Data Analyst,United Kingdom,2023,"nlp, computer vision, python, pytorch"
3,NLP Engineer,Brazil,2022,"data analysis, statistics, tensorflow, python"
4,AI Researcher,Netherlands,2022,"nlp, machine learning, pytorch, sql"
5,Prompt Engineer,Brazil,2021,"tensorflow, computer vision, nlp, machine learning"
6,AI Engineer,Singapore,2021,"statistics, python, tensorflow, machine learning"
7,AI Researcher,Brazil,2020,"computer vision, deep learning, tensorflow, pytorch"
8,Data Analyst,United States,2024,"statistics, machine learning, computer vision, sql"
9,NLP Engineer,Singapore,2020,"deep learning, machine learning, sql, pytorch"


## 11. Focus Maroc

On vérifie la présence du Maroc dans le dataset final.

In [15]:
df_morocco = df_clean[df_clean["country"] == "Morocco"]

print("Nombre d'offres au Maroc :", df_morocco.shape[0])

print("\nOffres Maroc par année :")
print(df_morocco["year"].value_counts().sort_index())

print("\nSources des offres Maroc :")
print(df_morocco["original_file"].value_counts())

print("\nTop titres Maroc :")
print(df_morocco["job_title"].value_counts().head(20))

Nombre d'offres au Maroc : 1058

Offres Maroc par année :
year
2023    909
2025      3
2026    146
Name: count, dtype: int64

Sources des offres Maroc :
original_file
huggingface_global_2023_data_jobs_skills_filtered.csv    909
jobs_raw.json                                             75
jobs_data_ai_clean.csv                                    74
Name: count, dtype: int64

Top titres Maroc :
job_title
Data Engineer                                                                     84
Data Scientist                                                                    42
Data Analyst                                                                      39
Senior Data Engineer                                                              23
Senior Data Scientist                                                             21
Data scientist                                                                    18
Data Engineer (H/F)                                                               16

## 12. Valeurs manquantes globales

Cette table sera utilisée dans le Data Quality Framework.

In [16]:
missing_table = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percent": (df_clean.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

missing_table

,missing_count,missing_percent
description,748740,99.59
job_url,743008,98.83
industry,630860,83.91
salary_currency,630308,83.84
salary,610543,81.21
job_type,130543,17.36
company_name,120030,15.97
education_required,9698,1.29
tools_used,8793,1.17
experience_level,1631,0.22


## 13. Vérification des doublons

On vérifie les doublons exacts, puis les doublons potentiels basés sur :

- titre du poste
- entreprise
- pays
- date de publication

In [17]:
exact_duplicates = df_clean.duplicated().sum()
print("Doublons exacts :", exact_duplicates)

potential_duplicates = df_clean.duplicated(
    subset=["job_title", "company_name", "country", "date_posted"],
    keep=False
).sum()

print("Doublons potentiels :", potential_duplicates)

if potential_duplicates > 0:
    display(df_clean[df_clean.duplicated(
        subset=["job_title", "company_name", "country", "date_posted"],
        keep=False
    )][["job_title", "company_name", "country", "date_posted", "original_file"]].head(20))

Doublons exacts : 0
Doublons potentiels : 56744


,job_title,company_name,country,date_posted,original_file
1,MLOps Engineer,NaN,India,2020-04-12,global_ai_jobs_dataset.csv
4,AI Researcher,NaN,Netherlands,2022-07-03,global_ai_jobs_dataset.csv
9,NLP Engineer,NaN,Singapore,2020-05-10,global_ai_jobs_dataset.csv
10,MLOps Engineer,NaN,Germany,2020-05-02,global_ai_jobs_dataset.csv
11,Data Analyst,NaN,Canada,2021-01-18,global_ai_jobs_dataset.csv
12,Data Analyst,NaN,Germany,2022-06-14,global_ai_jobs_dataset.csv
15,NLP Engineer,NaN,Australia,2020-04-18,global_ai_jobs_dataset.csv
16,Data Scientist,NaN,Japan,2022-03-25,global_ai_jobs_dataset.csv
21,Machine Learning Engineer,NaN,Germany,2020-01-27,global_ai_jobs_dataset.csv
26,Prompt Engineer,NaN,Japan,2021-09-28,global_ai_jobs_dataset.csv


## 14. Sauvegarde de la version vérifiée

On sauvegarde une version confirmée du dataset final propre.

In [18]:
OUTPUT_PATH = "final_data_ai_jobs_clean_2020_2026_verified.csv"

df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Fichier sauvegardé :", OUTPUT_PATH)
print("Shape :", df_clean.shape)

Fichier sauvegardé : final_data_ai_jobs_clean_2020_2026_verified.csv
Shape : (751801, 30)


## 15. Conclusion

Cette étape a permis de vérifier que le dataset final est bien harmonisé et exploitable.

Les points principaux sont :

- Les fichiers provenant de plusieurs sources ont été standardisés dans un schéma commun.
- Les dates, pays, compétences et statuts de télétravail ont été nettoyés.
- Les colonnes essentielles sont largement complètes.
- Le dataset couvre la période 2020–2026.
- Le dataset est déséquilibré, notamment à cause de la forte présence de la source Hugging Face 2023.
- Ce dataset propre servira aux étapes suivantes : Data Quality Framework, MongoDB, Web Content Mining, Graph Mining, prédiction et dashboard.